In [25]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)



Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [26]:
import pandas as pd
from datetime import datetime, timedelta
from collections import defaultdict
import pytz

# Define time period
# Calculate the start date (2 days ago) at 00:00:00 UTC
start_date = (datetime.now(pytz.UTC) - timedelta(days=2)).date()

# Format as 'YYYY-MM-DDT00:00:00Z'
start = f"{start_date}T00:00:00Z"

# List of owners to pull from
import urllib.parse

list_of_owners = ['HTOC Org']
final_results = []
type_names = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR", "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition})'
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)

        ro.set_http_method('GET')
        ro.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}&fields=tags,observations,associatedGroups&resultStart=0&resultLimit=10000'
        )

         # Send the request
        response = tc.api_request(ro)

        # Parse response
        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")

    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean results
if final_results:
    # Extract and normalize data only if 'data' key exists and contains 'summary'
    normalized_data = []
    for result in final_results:
        if 'data' in result:
            for item in result['data']:
                if 'summary' in item:
                    normalized_data.append(item)

    if normalized_data:
        observed_src = pd.json_normalize(normalized_data)
        observed_src['indicator'] = observed_src['summary'].str.split(' ', expand=True)[0].str.strip()
        observed_src = observed_src.drop_duplicates(subset='indicator', keep='first')  # Remove duplicates based on 'indicator'
        observed_src = observed_src[observed_src['lastObserved'] >= start]
        print(f"\nRetrieved {len(observed_src)} unique observed indicators.")
    else:
        print("\nNo valid indicators with 'summary' key retrieved.")
else:
    print("\nNo indicators retrieved.")




Querying owner: HTOC Org

Retrieved 1066 unique observed indicators.


In [27]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=3)

# Load observed data
observed_data_df = load_observed_data(file_paths)



Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250814.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250813.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250812.csv']


C:\Users\jaskew\AppData\Local\Temp\ipykernel_17580\564055639.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


Loaded data from 3 files.


In [28]:
import pandas as pd
from datetime import timedelta
import warnings

warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

# ── Time cutoffs ──────────────────────────────────────────────────────────────
cutoff = pd.Timestamp.utcnow()
date_added_cutoff = cutoff - timedelta(days=30)

# ── Collect filtered rows fast, then concat once ─────────────────────────────
all_filtered = []

print(f"observed_src shape: {observed_src.shape}")

# ── Loop through each row in observed_src ─────────────────────────────────────
for _, row in observed_src.iterrows():
    tags_data = row.get('tags.data')
    if not isinstance(tags_data, list):
        continue

    # Normalize the tags for the current row
    tags_df = pd.json_normalize(tags_data)

    # Ensure string and apply VA rename before filtering
    tags_df['name'] = (
        tags_df['name']
        .astype(str)
        .str.replace('VA CSOC CTS Splunk', 'VA Splunk API', regex=False)
    )

    # Skip if all associated groups are Adversary
    if 'associatedGroups.data' in observed_src.columns:
        ag_data = row.get('associatedGroups.data')
        if isinstance(ag_data, list) and len(ag_data) > 0:
            groups_df = pd.json_normalize(ag_data)
            if 'type' in groups_df.columns and set(groups_df['type']) == {'Adversary'}:
                continue

    # All tag names (for all_tags)
    all_tags_list = tags_df['name'].tolist()

    # Filter for API tags
    api_tags = tags_df[tags_df['name'].str.contains('API', case=False, na=False)].copy()
    if api_tags.empty:
        continue

    # Add metadata columns (including rating, confidence, id)
    meta_cols = [
        'summary', 'observations', 'description', 'type',
        'dateAdded', 'lastModified', 'lastObserved', 'webLink',
        'rating', 'confidence', 'id'
    ]
    for col in meta_cols:
        api_tags[col] = row.get(col)

    # Attach all tags list
    api_tags['all_tags'] = [all_tags_list] * len(api_tags)

    # Accumulate
    all_filtered.append(api_tags)

# ── Create filtered_tags DataFrame ────────────────────────────────────────────
filtered_tags = pd.concat(all_filtered, ignore_index=True) if all_filtered else pd.DataFrame()
print(f"filtered_tags shape: {filtered_tags.shape}")

# Ensure datetimes
if not filtered_tags.empty:
    filtered_tags['lastObserved'] = pd.to_datetime(filtered_tags['lastObserved'], errors='coerce', utc=True)
    filtered_tags['dateAdded'] = pd.to_datetime(filtered_tags['dateAdded'], errors='coerce', utc=True)

# ── Validate observed_data_df has needed columns ──────────────────────────────
required_columns = ['indicator', 'OpDiv', 'obs_date']
missing_columns = [c for c in required_columns if c not in observed_data_df.columns]
if missing_columns:
    raise ValueError(f"Missing columns in ProcessedObservedData.csv: {missing_columns}")

print(f"observed_data_df shape: {observed_data_df.shape}")

# ── Clean name -> match OpDiv (remove trailing ' Splunk API') ─────────────────
if not filtered_tags.empty:
    filtered_tags['cleaned_name'] = filtered_tags['name'].str.replace(r'\s+Splunk API$', '', regex=True)

    # Map observed_date by (indicator=summary, OpDiv=cleaned_name)
    filtered_tags['observed_date'] = pd.NaT
    # Ensure observed_data_df obs_date is datetime (naive)
    observed_data_df['obs_date'] = pd.to_datetime(observed_data_df['obs_date'], errors='coerce')

    for idx, r in filtered_tags.iterrows():
        summary = r.get('summary')
        cleaned_name = r.get('cleaned_name')
        if pd.isna(summary) or pd.isna(cleaned_name):
            continue
        match = observed_data_df[
            (observed_data_df['indicator'] == summary) &
            (observed_data_df['OpDiv'] == cleaned_name)
        ]
        if not match.empty:
            filtered_tags.at[idx, 'observed_date'] = match['obs_date'].iloc[0]

    filtered_tags['observed_date'] = pd.to_datetime(filtered_tags['observed_date'], errors='coerce')

    # Drop helper
    filtered_tags.drop(columns=['cleaned_name'], inplace=True, errors='ignore')
    print(f"filtered_tags shape after observed_date mapping: {filtered_tags.shape}")

# ── Recent filters ────────────────────────────────────────────────────────────
# Last 24h by lastObserved (aware)
recent_tags = filtered_tags[filtered_tags['lastObserved'] >= cutoff - timedelta(hours=24)].copy()
print(f"recent_tags shape after lastObserved filter: {recent_tags.shape}")

# Make cutoff naive to compare against observed_date (naive)
cutoff_naive = cutoff.tz_convert(None)

# Last 2 days by observed_date (naive)
recent_tags = filtered_tags[filtered_tags['observed_date'] >= cutoff_naive - timedelta(days=2)].copy()
print(f"recent_tags shape after observed_date filter: {recent_tags.shape}")

# ── Partner extraction and grouping ───────────────────────────────────────────
recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)
print(f"recent_tags shape after partner column: {recent_tags.shape}")

partner_groups = (
    recent_tags.groupby('summary')['partner']
    .agg(['nunique', lambda s: ', '.join(sorted(set(s.dropna())))]).reset_index()
    .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
)
print(f"partner_groups shape: {partner_groups.shape}")

recent_tags = recent_tags.merge(partner_groups, on='summary', how='left')
print(f"recent_tags shape after merge: {recent_tags.shape}")

# ── Additional filters ────────────────────────────────────────────────────────
recent_tags = recent_tags[recent_tags['partner_count'] >= 2]
print(f"recent_tags shape after partner_count filter: {recent_tags.shape}")

recent_tags = recent_tags[recent_tags['observations'] < 15000]
print(f"recent_tags shape after observations filter: {recent_tags.shape}")

recent_tags = recent_tags[recent_tags['dateAdded'] >= date_added_cutoff]
print(f"recent_tags shape after dateAdded filter: {recent_tags.shape}")

# Deduplicate by summary
recent_tags = recent_tags.drop_duplicates(subset='summary', keep='first')
print(f"recent_tags shape after deduplication: {recent_tags.shape}")

# Drop unneeded columns if present
cols_to_drop = [
    'techniqueId', 'tactics.data', 'tactics.count',
    'platforms.data', 'platforms.count', 'partner', 'name'
]
recent_tags = recent_tags.drop(columns=[c for c in cols_to_drop if c in recent_tags.columns], errors='ignore')
print(f"recent_tags shape after dropping columns: {recent_tags.shape}")

# Remove rows where all_tags contains unwanted markers
if 'all_tags' in recent_tags.columns:
    recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: isinstance(x, list) and 'I&W' in x)]
    print(f"recent_tags shape after I&W filter: {recent_tags.shape}")
    recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: isinstance(x, list) and 'htoc_wl' in x)]
    print(f"recent_tags shape after htoc_wl filter: {recent_tags.shape}")

# Preview
print(f"recent_tags final shape: {recent_tags.shape}")
recent_tags


observed_src shape: (1066, 33)
filtered_tags shape: (7380, 19)
observed_data_df shape: (13657, 7)
filtered_tags shape after observed_date mapping: (7380, 20)
recent_tags shape after lastObserved filter: (6658, 20)
recent_tags shape after observed_date filter: (4834, 20)
recent_tags shape after partner column: (4834, 21)
partner_groups shape: (985, 3)
recent_tags shape after merge: (4834, 23)
recent_tags shape after partner_count filter: (4700, 23)
recent_tags shape after observations filter: (687, 23)
recent_tags shape after dateAdded filter: (268, 23)
recent_tags shape after deduplication: (62, 23)
recent_tags shape after dropping columns: (62, 16)
recent_tags shape after I&W filter: (62, 16)
recent_tags shape after htoc_wl filter: (62, 16)
recent_tags final shape: (62, 16)


,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,rating,confidence,all_tags,observed_date,partner_count,partners
3,6755399465469202,2025-08-14T10:41:27Z,INC9172314,159.203.36.189,9702,Address,2025-07-31 12:06:29+00:00,2025-08-14T14:05:41Z,2025-08-14 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,78.0,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-08-14,4,"CMS, FDA, IHS, OS"
11,6755399467420964,2025-08-14T10:41:27Z,RITM0606742,64.62.156.55,11601,Address,2025-08-13 15:08:18+00:00,2025-08-14T14:04:39Z,2025-08-14 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,80.0,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-08-14,4,"CMS, FDA, IHS, OS"
15,6755399467420975,2025-08-14T10:41:27Z,RITM0606742,45.141.56.48,91,Address,2025-08-13 15:08:24+00:00,2025-08-14T12:24:25Z,2025-08-14 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,80.0,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-08-14,3,"CMS, OS, VA"
18,6755399467420966,2025-08-14T14:06:23Z,RITM0606742,65.49.1.97,573,Address,2025-08-13 15:08:20+00:00,2025-08-14T12:24:00Z,2025-08-14 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,80.0,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-08-14,5,"CMS, DHA, IHS, OS, VA"
23,6755399467420983,2025-08-14T10:41:27Z,RITM0606742,65.49.1.173,1063,Address,2025-08-13 15:08:31+00:00,2025-08-14T12:23:52Z,2025-08-14 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,80.0,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-08-14,3,"CMS, IHS, OS"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2999,5629499560028011,2025-08-14T10:41:27Z,INC9152937,218.3.12.22,6396,Address,2025-07-18 12:38:39+00:00,2025-08-11T18:43:33Z,2025-08-14 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,77.0,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-08-14,4,"CMS, FDA, HHS, OS"
3231,6755399465229474,2025-08-14T10:41:27Z,TASK0902923,199.45.154.143,1217,Address,2025-07-28 17:31:26+00:00,2025-08-11T07:42:57Z,2025-08-14 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,78.0,"[DHA Splunk API, OS Splunk API, CMS Splunk API...",2025-08-13,3,"IHS, OS, VA"
3544,5629499561376898,2025-08-14T10:41:27Z,TASK0902923,206.168.34.44,1106,Address,2025-07-28 17:34:01+00:00,2025-08-11T07:42:56Z,2025-08-14 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,78.0,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-08-14,5,"CMS, IHS, NIH, OS, VA"
4790,6755399467089130,2025-08-14T10:41:27Z,INC9186882,103.124.94.57,3360,Address,2025-08-09 18:13:37+00:00,2025-08-11T00:32:12Z,2025-08-14 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,80.0,"[OS Splunk API, FDA Splunk API, suspicious, CM...",2025-08-14,3,"CMS, FDA, OS"


In [29]:
exclude_list = [
    '159.203.36.189', '88.218.193.157', '64.62.156.55', '45.141.56.48',
    '65.49.1.97', '65.49.1.173', '65.49.1.141', '65.49.1.59', '65.49.1.94',
    '65.49.1.13', '64.62.156.122', '45.230.66.97', '65.49.1.19', '64.62.156.220',
    '65.49.1.96', '65.49.1.186', '64.62.156.141', '64.62.156.177', '65.49.1.122',
    '64.62.156.204', '45.230.66.106', '64.62.156.25', '45.230.66.115',
    '64.62.156.194', '65.49.1.194', '65.49.1.87', '64.62.156.139', '45.230.66.99',
    '64.62.156.136', '65.49.1.83', '64.62.156.75', '64.62.156.104',
    '64.62.156.111', '64.62.156.50', '65.49.1.165', '64.62.156.229',
    '45.230.66.110', '64.62.156.166', '65.49.1.134', '65.49.1.89', '64.62.156.20',
    '190.14.253.70', '190.14.253.67', '190.14.253.68', '190.14.253.69',
    '190.14.253.66', '64.62.156.169', '65.49.1.37', '121.152.230.177',
    '125.88.174.211', '51.15.187.166', '204.76.203.61', '198.55.98.80',
    '203.56.198.45', '114.98.227.36', '45.78.192.92', '166.186.196.135',
    '218.3.12.22', '199.45.154.143', '206.168.34.44', '103.124.94.57',
    '14.18.37.109', '45.78.192.123'
]

check_recent_tags = recent_tags[~recent_tags['summary'].isin(exclude_list)]
check_recent_tags

,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,rating,confidence,all_tags,observed_date,partner_count,partners
622,6755399467419425,2025-08-14T12:24:25Z,NaN,service@net-star.net,10,EmailAddress,2025-08-13 11:08:17+00:00,2025-08-14T02:23:44Z,2025-08-14 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,90.0,"[FDA Splunk API, Telegram, CMS Splunk API, Obs...",2025-08-14,2,"CMS, FDA"


In [30]:
import pandas as pd
from datetime import timedelta
import warnings

warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

# ── Time cutoffs ──────────────────────────────────────────────────────────────
cutoff = pd.Timestamp.utcnow()
date_added_cutoff = cutoff - timedelta(days=30)

# ── Collect filtered rows fast, then concat once ─────────────────────────────
all_filtered = []

print(f"observed_src shape: {observed_src.shape}")

# ── Loop through each row in observed_src ─────────────────────────────────────
for _, row in observed_src.iterrows():
    tags_data = row.get('tags.data')
    if not isinstance(tags_data, list):
        continue

    # Normalize the tags for the current row
    tags_df = pd.json_normalize(tags_data)

    # Ensure string and apply VA rename before filtering
    tags_df['name'] = (
        tags_df['name']
        .astype(str)
        .str.replace('VA CSOC CTS Splunk', 'VA Splunk API', regex=False)
    )

    # Skip if all associated groups are Adversary
    if 'associatedGroups.data' in observed_src.columns:
        ag_data = row.get('associatedGroups.data')
        if isinstance(ag_data, list) and len(ag_data) > 0:
            groups_df = pd.json_normalize(ag_data)
            if 'type' in groups_df.columns and set(groups_df['type']) == {'Adversary'}:
                continue

    # All tag names (for all_tags)
    all_tags_list = tags_df['name'].tolist()

    # Filter for API tags
    api_tags = tags_df[tags_df['name'].str.contains('API', case=False, na=False)].copy()
    if api_tags.empty:
        continue

    # Add metadata columns (including rating, confidence, id)
    meta_cols = [
        'summary', 'observations', 'description', 'type',
        'dateAdded', 'lastModified', 'lastObserved', 'webLink',
        'rating', 'confidence', 'id'
    ]
    for col in meta_cols:
        api_tags[col] = row.get(col)

    # Attach all tags list
    api_tags['all_tags'] = [all_tags_list] * len(api_tags)

    # Accumulate
    all_filtered.append(api_tags)

# ── Create filtered_tags DataFrame ────────────────────────────────────────────
filtered_tags = pd.concat(all_filtered, ignore_index=True) if all_filtered else pd.DataFrame()
print(f"filtered_tags shape: {filtered_tags.shape}")

# Ensure datetimes
if not filtered_tags.empty:
    filtered_tags['lastObserved'] = pd.to_datetime(filtered_tags['lastObserved'], errors='coerce', utc=True)
    filtered_tags['dateAdded'] = pd.to_datetime(filtered_tags['dateAdded'], errors='coerce', utc=True)

# ── Validate observed_data_df has needed columns ──────────────────────────────
required_columns = ['indicator', 'OpDiv', 'obs_date']
missing_columns = [c for c in required_columns if c not in observed_data_df.columns]
if missing_columns:
    raise ValueError(f"Missing columns in ProcessedObservedData.csv: {missing_columns}")

print(f"observed_data_df shape: {observed_data_df.shape}")

# ── Clean name -> match OpDiv (remove trailing ' Splunk API') ─────────────────
if not filtered_tags.empty:
    filtered_tags['cleaned_name'] = filtered_tags['name'].str.replace(r'\s+Splunk API$', '', regex=True)

    # Map observed_date by (indicator=summary, OpDiv=cleaned_name)
    filtered_tags['observed_date'] = pd.NaT
    # Ensure observed_data_df obs_date is datetime (naive)
    observed_data_df['obs_date'] = pd.to_datetime(observed_data_df['obs_date'], errors='coerce')

    for idx, r in filtered_tags.iterrows():
        summary = r.get('summary')
        cleaned_name = r.get('cleaned_name')
        if pd.isna(summary) or pd.isna(cleaned_name):
            continue
        match = observed_data_df[
            (observed_data_df['indicator'] == summary) &
            (observed_data_df['OpDiv'] == cleaned_name)
        ]
        if not match.empty:
            filtered_tags.at[idx, 'observed_date'] = match['obs_date'].iloc[0]

    filtered_tags['observed_date'] = pd.to_datetime(filtered_tags['observed_date'], errors='coerce')

    # Drop helper
    filtered_tags.drop(columns=['cleaned_name'], inplace=True, errors='ignore')
    print(f"filtered_tags shape after observed_date mapping: {filtered_tags.shape}")

# ── Recent filters ────────────────────────────────────────────────────────────
# Last 24h by lastObserved (aware)
recent_tags = filtered_tags[filtered_tags['lastObserved'] >= cutoff - timedelta(hours=24)].copy()
print(f"recent_tags shape after lastObserved filter: {recent_tags.shape}")

# Make cutoff naive to compare against observed_date (naive)
cutoff_naive = cutoff.tz_convert(None)

# Last 2 days by observed_date (naive)
recent_tags = filtered_tags[filtered_tags['observed_date'] >= cutoff_naive - timedelta(days=2)].copy()
print(f"recent_tags shape after observed_date filter: {recent_tags.shape}")

# ── Partner extraction and grouping ───────────────────────────────────────────
recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)
print(f"recent_tags shape after partner column: {recent_tags.shape}")

partner_groups = (
    recent_tags.groupby('summary')['partner']
    .agg(['nunique', lambda s: ', '.join(sorted(set(s.dropna())))]).reset_index()
    .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
)
print(f"partner_groups shape: {partner_groups.shape}")

recent_tags = recent_tags.merge(partner_groups, on='summary', how='left')
print(f"recent_tags shape after merge: {recent_tags.shape}")

# ── Additional filters ────────────────────────────────────────────────────────
recent_tags = recent_tags[recent_tags['partner_count'] >= 2]
print(f"recent_tags shape after partner_count filter: {recent_tags.shape}")

recent_tags = recent_tags[recent_tags['observations'] < 15000]
print(f"recent_tags shape after observations filter: {recent_tags.shape}")

recent_tags = recent_tags[recent_tags['dateAdded'] >= date_added_cutoff]
print(f"recent_tags shape after dateAdded filter: {recent_tags.shape}")

# Deduplicate by summary
recent_tags = recent_tags.drop_duplicates(subset='summary', keep='first')
print(f"recent_tags shape after deduplication: {recent_tags.shape}")

# Drop unneeded columns if present
cols_to_drop = [
    'techniqueId', 'tactics.data', 'tactics.count',
    'platforms.data', 'platforms.count', 'partner', 'name'
]
recent_tags = recent_tags.drop(columns=[c for c in cols_to_drop if c in recent_tags.columns], errors='ignore')
print(f"recent_tags shape after dropping columns: {recent_tags.shape}")

# Remove rows where all_tags contains unwanted markers
if 'all_tags' in recent_tags.columns:
    recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: isinstance(x, list) and 'I&W' in x)]
    print(f"recent_tags shape after I&W filter: {recent_tags.shape}")
    recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: isinstance(x, list) and 'htoc_wl' in x)]
    print(f"recent_tags shape after htoc_wl filter: {recent_tags.shape}")

# Preview
print(f"recent_tags final shape: {recent_tags.shape}")
recent_tags.head(20)


observed_src shape: (1066, 33)
filtered_tags shape: (7380, 19)
observed_data_df shape: (13657, 7)
filtered_tags shape after observed_date mapping: (7380, 20)
recent_tags shape after lastObserved filter: (6658, 20)
recent_tags shape after observed_date filter: (4834, 20)
recent_tags shape after partner column: (4834, 21)
partner_groups shape: (985, 3)
recent_tags shape after merge: (4834, 23)
recent_tags shape after partner_count filter: (4700, 23)
recent_tags shape after observations filter: (687, 23)
recent_tags shape after dateAdded filter: (268, 23)
recent_tags shape after deduplication: (62, 23)
recent_tags shape after dropping columns: (62, 16)
recent_tags shape after I&W filter: (62, 16)
recent_tags shape after htoc_wl filter: (62, 16)
recent_tags final shape: (62, 16)


,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,rating,confidence,all_tags,observed_date,partner_count,partners
3,6755399465469202,2025-08-14T10:41:27Z,INC9172314,159.203.36.189,9702,Address,2025-07-31 12:06:29+00:00,2025-08-14T14:05:41Z,2025-08-14 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,78.0,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-08-14,4,"CMS, FDA, IHS, OS"
11,6755399467420964,2025-08-14T10:41:27Z,RITM0606742,64.62.156.55,11601,Address,2025-08-13 15:08:18+00:00,2025-08-14T14:04:39Z,2025-08-14 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,80.0,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-08-14,4,"CMS, FDA, IHS, OS"
15,6755399467420975,2025-08-14T10:41:27Z,RITM0606742,45.141.56.48,91,Address,2025-08-13 15:08:24+00:00,2025-08-14T12:24:25Z,2025-08-14 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,80.0,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-08-14,3,"CMS, OS, VA"
18,6755399467420966,2025-08-14T14:06:23Z,RITM0606742,65.49.1.97,573,Address,2025-08-13 15:08:20+00:00,2025-08-14T12:24:00Z,2025-08-14 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,80.0,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-08-14,5,"CMS, DHA, IHS, OS, VA"
23,6755399467420983,2025-08-14T10:41:27Z,RITM0606742,65.49.1.173,1063,Address,2025-08-13 15:08:31+00:00,2025-08-14T12:23:52Z,2025-08-14 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,80.0,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-08-14,3,"CMS, IHS, OS"
26,5629499563360158,2025-08-14T14:06:23Z,RITM0606742,65.49.1.141,1074,Address,2025-08-13 15:08:26+00:00,2025-08-14T12:23:52Z,2025-08-14 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,80.0,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-08-14,5,"CMS, DHA, IHS, OS, VA"
31,5629499563360174,2025-08-14T14:06:23Z,RITM0606742,65.49.1.59,1250,Address,2025-08-13 15:08:43+00:00,2025-08-14T12:23:50Z,2025-08-14 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,80.0,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-08-14,5,"CMS, DHA, IHS, OS, VA"
36,5629499563360175,2025-08-14T14:06:23Z,RITM0606742,65.49.1.94,1611,Address,2025-08-13 15:08:46+00:00,2025-08-14T12:23:49Z,2025-08-14 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,80.0,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-08-13,5,"CMS, DHA, IHS, OS, VA"
41,6755399467420995,2025-08-14T10:41:27Z,RITM0606742,65.49.1.13,1523,Address,2025-08-13 15:08:37+00:00,2025-08-14T12:23:47Z,2025-08-14 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,80.0,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-08-14,4,"CMS, IHS, OS, VA"
45,6755399467421000,2025-08-14T14:06:23Z,RITM0606742,64.62.156.122,1862,Address,2025-08-13 15:08:41+00:00,2025-08-14T12:23:44Z,2025-08-14 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,80.0,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-08-14,5,"CMS, DHA, IHS, OS, VA"


In [31]:
import pandas as pd
import urllib.parse

# Extract unique indicators from recent_tags
indicators = recent_tags['summary'].unique()

# Initialize a list to store attribute data
attributes_data = []

# Track indicators with no attributes
indicators_with_no_attributes = []

# Iterate over unique indicators
for indicator in indicators:
    try:
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators/{indicator}?fields=attributes&resultStart=0&resultLimit=1000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            data = response.json().get('data', {})
            attributes = data.get('attributes', {}).get('data', [])

            if not attributes:
                print(f"No attributes found for indicator: {indicator}")
                # Track indicators with no attributes
                indicators_with_no_attributes.append(indicator)
            else:
                for attr in attributes:
                    attr.update({
                        'id': data.get('id'),
                        'summary': data.get('summary'),
                        'type': data.get('type'),
                        'ownerName': data.get('ownerName')
                    })
                    attributes_data.append(attr)
        # Suppress non-JSON and known 400 error responses
    except Exception as e:
        # Suppress error output, including known 400 error
        if hasattr(e, 'response') and getattr(e.response, 'status_code', None) == 400:
            continue
        if "Status Code: 400" in str(e):
            continue
        pass

# Create attributes 
attributes_observed_src = pd.json_normalize(attributes_data)

# Un-nest 'createdBy' and filter out 'SOAR' entries
if not attributes_observed_src.empty and attributes_observed_src['createdBy.lastName'].notnull().any():
    attributes_observed_src = attributes_observed_src[attributes_observed_src['createdBy.lastName'] != 'SOAR']

# Drop duplicates based on 'id' to keep unique attribute records
attributes_observed_src = attributes_observed_src.drop_duplicates(subset='id').reset_index(drop=True)

# Filter recent_tags for indicators that had attributes
filtered_with_attrs = recent_tags[recent_tags['summary'].isin(attributes_observed_src['summary'])]

# Filter recent_tags for indicators that had no attributes
no_attrs_df = recent_tags[recent_tags['summary'].isin(indicators_with_no_attributes)]

# Combine both and remove duplicates based on 'summary'
filtered_recent_tags = pd.concat([filtered_with_attrs, no_attrs_df], ignore_index=True)
filtered_recent_tags = filtered_recent_tags.drop_duplicates(subset='summary').reset_index(drop=True)


No attributes found for indicator: service@net-star.net


In [32]:
filtered_recent_tags

,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,rating,confidence,all_tags,observed_date,partner_count,partners
0,6755399467419425,2025-08-14T12:24:25Z,NaN,service@net-star.net,10,EmailAddress,2025-08-13 11:08:17+00:00,2025-08-14T02:23:44Z,2025-08-14 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,90.0,"[FDA Splunk API, Telegram, CMS Splunk API, Obs...",2025-08-14,2,"CMS, FDA"


In [62]:
import pandas as pd
import requests
import time
import ipaddress
from datetime import datetime
import concurrent.futures

# API Keys
VT_API_KEY = "a8b3e24dbd2e6c0cb002784aeb8fee6217a6a425cb11ddf9a3d3361281fbbb08"
OTX_API_KEY = "ea83cf4792fc5db7acc741e82286c0b717fc99be98ec5b14de7e63cd3f74bcfe"

# Headers for API requests
VT_HEADERS = {"x-apikey": VT_API_KEY}
OTX_HEADERS = {"X-OTX-API-KEY": OTX_API_KEY}

# API URLs
VT_URL_IP = "https://www.virustotal.com/api/v3/ip_addresses/{}"
VT_URL_DOMAIN = "https://www.virustotal.com/api/v3/domains/{}"
OTX_URL_IP = "https://otx.alienvault.io/api/v1/indicators/IPv4/{}/general"
OTX_URL_DOMAIN = "https://otx.alienvault.io/api/v1/indicators/domain/{}/general"
OTX_URL_HOSTNAME = "https://otx.alienvault.io/api/v1/indicators/hostname/{}"

def is_ip(value):
    """ Check if the given value is a valid IP address. """
    try:
        ipaddress.ip_address(value)
        return True
    except ValueError:
        return False

def determine_query_type(query):
    """ Determine if the query is an IP, domain, or hostname. """
    if is_ip(query):
        return "ip"
    elif "." in query:
        return "hostname"
    else:
        return "domain"

def fetch_virustotal_data(query):
    """Fetch data from VirusTotal API for IP or Domain using ThreatConnect enrich endpoint."""
    import urllib.parse

    # indicator_values should be a list of indicator values to enrich
    # For compatibility, if query is a single value, wrap in list
    indicator_values = [query] if isinstance(query, str) else query
    enriched_results = []

    for value in indicator_values:
        try:
            # Use the indicator *value*, not the ID
            encoded_value = urllib.parse.quote(value)
            enrich_url = f'/v3/indicators/{encoded_value}/enrich?type=Shodan&type=VirusTotalV3'
            ro.set_http_method('POST')
            ro.set_request_uri(enrich_url)
            ro.set_body({})
            enrich_response = tc.api_request(ro)

            if enrich_response.status_code == 200:
                enrich_data = enrich_response.json()
                enrich_data['summary'] = value  # Save the value as key
                enriched_results.append(enrich_data)
        except Exception as e:
            continue
    # Unnest the 'data' field in enriched_results if it's a list of dicts
    if enriched_results:
        # If enriched_results is a list of dicts with 'data' key, flatten each
        flattened = []
        for item in enriched_results:
            if isinstance(item, dict) and 'data' in item:
                flat = item.copy()
                flat.update(pd.json_normalize(item['data']).iloc[0].to_dict() if isinstance(item['data'], dict) else {})
                del flat['data']
                flattened.append(flat)
            else:
                flattened.append(item)
        enriched_results = flattened

    # Return first result if only one value, else list
    if len(enriched_results) == 1:
        return enriched_results[0]
    
    return enriched_results

def fetch_otx_data(query):
    """Fetch data from OTX API for IP, Domain, or Hostname."""
    query_type = determine_query_type(query)

    if query_type == "ip":
        url = OTX_URL_IP.format(query)
    elif query_type == "hostname":
        url = OTX_URL_HOSTNAME.format(query)
    else:
        url = OTX_URL_DOMAIN.format(query)

    try:
        response = requests.get(url, headers=OTX_HEADERS, verify=False)
        response.raise_for_status()
        data = response.json()

        return {
            "search_term": query,
            "base_indicator": data.get('base_indicator', {}),
            "reputation": data.get('reputation'),
            "geo": data.get('geo', {}),
            "whois": data.get('whois', {}),
            "open_ports": data.get('open_ports', []),
            "link": f"https://otx.alienvault.com/indicator/{query_type}/{query}"
        }

    except Exception as e:
        print(f"OTX Error for {query}: {e}")
        return None

def unnest_base_indicator(df):
    """ Unnest the 'base_indicator' column and create separate columns for its keys. """
    if 'base_indicator' not in df.columns:
        return df

    base_df = pd.json_normalize(df['base_indicator'])
    base_df.columns = [f"base_{col}" for col in base_df.columns]

    df = df.drop(columns=['base_indicator']).reset_index(drop=True)
    df = pd.concat([df, base_df], axis=1)
    return df

def process_indicator(indicator, observed_by, partner_count):
    """Fetch data for a single indicator."""
    timestamp = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

    with concurrent.futures.ThreadPoolExecutor() as executor:
        vt_future = executor.submit(fetch_virustotal_data, indicator)
        otx_future = executor.submit(fetch_otx_data, indicator)

        vt_data = vt_future.result()
        otx_data = otx_future.result()

    if vt_data:
        vt_data.update({
            "timestamp": timestamp,
            "observed_by": observed_by,
            "partner_count": partner_count
        })

    if otx_data:
        otx_data.update({
            "timestamp": timestamp,
            "observed_by": observed_by,
            "partner_count": partner_count
        })

    return vt_data, otx_data

def main(recent_tags):
    """ Main function to process indicators. """
    if 'summary' not in recent_tags.columns:
        print("The 'summary' column is missing.")
        return pd.DataFrame(), pd.DataFrame()

    search_terms = recent_tags['summary'].dropna().unique().tolist()
    print(f"Processing {len(search_terms)} unique search terms.")

    vt_results = []
    otx_results = []

    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = []
        for indicator in search_terms:
            partners = recent_tags.loc[recent_tags['summary'] == indicator, 'partners'].values
            partner_count = recent_tags.loc[recent_tags['summary'] == indicator, 'partner_count'].values
            observed_by = partners[0] if len(partners) > 0 else "N/A"
            partner_count = partner_count[0] if len(partner_count) > 0 else "N/A"

            futures.append(executor.submit(process_indicator, indicator, observed_by, partner_count))

        for future in concurrent.futures.as_completed(futures):
            vt_data, otx_data = future.result()
            if vt_data:
                vt_results.append(vt_data)
            if otx_data:
                otx_results.append(otx_data)

    vt_df = pd.DataFrame(vt_results)
    otx_df = pd.DataFrame(otx_results)

    
    otx_df = unnest_base_indicator(otx_df)

    return vt_df, otx_df

if __name__ == "__main__":
    vt_df, otx_df = main(filtered_recent_tags)
    print("Script completed successfully.")


Processing 1 unique search terms.


C:\Users\jaskew\AppData\Local\Temp\ipykernel_17580\2999662670.py:129: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The EmailAddress service@net-star.net cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'


OTX Error for service@net-star.net: 400 Client Error: Bad Request for url: https://otx.alienvault.io/api/v1/indicators/hostname/service@net-star.net
Script completed successfully.


In [63]:
vt_df

""


In [64]:
otx_df

""


In [61]:
import os
import pandas as pd
import docx
from datetime import datetime
from docx import Document
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.oxml.shared import OxmlElement

# File paths
#TEMPLATE_PATH = r"z:\HTOC\HTOC Reports\I&W Reports\5. I&W Staging\I&W Report Template.docx"
TEMPLATE_PATH = r"C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\I&W Report Template.docx"
#OUTPUT_DIR = r"z:\HTOC\HTOC Reports\I&W Reports\5. I&W Staging\Generated Reports"
OUTPUT_DIR = r"C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports"

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

def consolidate_sources(vt_df, otx_df):
    """ Consolidate links from both DataFrames for each search term. """
    # Collect links from VirusTotal
    vt_links = vt_df.groupby('search_term')['link'].apply(lambda x: ', '.join(x.dropna())).reset_index()
    vt_links.columns = ['search_term', 'vt_links']

    # Collect links from OTX
    otx_links = otx_df.groupby('search_term')['link'].apply(lambda x: ', '.join(x.dropna())).reset_index()
    otx_links.columns = ['search_term', 'otx_links']

    # Merge the two link sets
    consolidated = pd.merge(vt_links, otx_links, on='search_term', how='outer')

    # Combine the links, handling NaN values
    consolidated['sources'] = consolidated[['vt_links', 'otx_links']].fillna('').apply(
        lambda x: ', '.join(filter(None, x)), axis=1
    )

    return consolidated[['search_term', 'sources']]

def extract_date(timestamp):
    """ Extract only the date from the timestamp. """
    if pd.isna(timestamp) or timestamp == 'N/A':
        return 'N/A'
    
    # Handle datetime object or string
    try:
        # Attempt to parse as a datetime object
        if isinstance(timestamp, str):
            timestamp = datetime.strptime(timestamp, "%Y-%m-%d %H:%M:%S")
        return timestamp.strftime("%Y-%m-%d")
    except Exception:
        return 'N/A'

def populate_table(table, data):
    """ Populate a Word table with the given data. """
    # Iterate over data and populate rows
    for index, row in data.iterrows():
        # Insert a new row before the last row (template row)
        new_row = table.add_row().cells
        new_row[0].text = str(row.get('search_term', 'N/A'))
        new_row[1].text = str(row.get('type', 'N/A'))
        new_row[2].text = extract_date(row.get('observed_date', 'N/A'))
        # For the 'observed_by_otx' column, stack values instead of comma-separating
        # observed_by_otx: stack values (one per line) from OTX and VirusTotal "observed_by" columns if present
        observed_by_otx = ''
        observed_by_list = []

        # Try to get observed_by from OTX
        if 'observed_by_otx' in row and pd.notna(row['observed_by_otx']):
            observed_by_list.extend([v.strip() for v in str(row['observed_by_otx']).split(',') if v.strip()])
        elif 'observed_by' in row and pd.notna(row['observed_by']):
            observed_by_list.extend([v.strip() for v in str(row['observed_by']).split(',') if v.strip()])

        # Remove duplicates and join with newlines
        if observed_by_list:
            observed_by_otx = '\n'.join(sorted(set(observed_by_list)))
        else:
            observed_by_otx = 'N/A'
        new_row[3].text = str(observed_by_otx)
        new_row[4].text = str(row.get('notes', ''))
        
def fill_word_template(template_path, output_path, df):
    """ Fill the template with data and place sources outside the table. """
    if not os.path.exists(template_path):
        print(f"Template not found: {template_path}")
        return
    
    try:
        doc = Document(template_path)

        # Populate the table
        table = None
        for tbl in doc.tables:
            if "Indicators/Identifiers" in tbl.rows[0].cells[0].text:
                table = tbl
                break

        if table:
            populate_table(table, df)

        # Find and replace `{{sources}}` placeholder outside the table
        for para in doc.paragraphs:
            if "{{ipaddress}}" in para.text:
                # Try to get the first search_term as the IP address
                ip_address = str(df['search_term'].iloc[0]) if 'search_term' in df.columns else 'N/A'
                para.text = para.text.replace("{{ipaddress}}", ip_address)
            if "{{asn}}" in para.text:
                # Try to get ASN from vt_df if available, else use 'N/A'
                asn_value = str(df['asn'].iloc[0]) if 'asn' in df.columns and not df['asn'].isna().all() else 'N/A'
                para.text = para.text.replace("{{asn}}", asn_value)
            if "{{whois}}" in para.text:
                # Try to get WHOIS info from otx_df if available, else use 'N/A'
                whois_value = str(df['whois'].iloc[0]) if 'whois' in df.columns and not df['whois'].isna().all() else 'N/A'
                para.text = para.text.replace("{{whois}}", whois_value)
            if "{{partners}}" in para.text:
                # Try to get partners from recent_tags if available, else use 'N/A'
                partners_value = ''
                if 'search_term' in df.columns and not df.empty:
                    search_term = df['search_term'].iloc[0]
                    partners_row = recent_tags[recent_tags['summary'] == search_term]
                    if not partners_row.empty and 'partners' in partners_row.columns:
                        partners_value = str(partners_row['partners'].iloc[0])
                if not partners_value:
                    partners_value = 'N/A'
                para.text = para.text.replace("{{partners}}", partners_value)
            if "{{weblink}}" in para.text:
                weblink_value = ''
                # Try to get the first search_term as the indicator
                weblink_value = ''
                if 'search_term' in df.columns and not df.empty:
                    search_term = df['search_term'].iloc[0]
                    # Try to find a 'webLink' in df for the indicator (if present)
                    if 'webLink' in df.columns:
                        match = df[df['search_term'] == search_term]
                        if not match.empty and pd.notna(match['webLink'].iloc[0]):
                            weblink_value = str(match['webLink'].iloc[0])
                    # Fallback: try 'link' column if 'webLink' is not present or empty
                    if not weblink_value and 'link' in df.columns:
                        match = df[df['search_term'] == search_term]
                        if not match.empty and pd.notna(match['link'].iloc[0]):
                            weblink_value = str(match['link'].iloc[0])
                para.text = para.text.replace("{{weblink}}", "")
                if weblink_value:
                    # Add hyperlink using WordprocessingML (only show the link, no display text)
                    r_id = doc.part.relate_to(
                        weblink_value, docx.opc.constants.RELATIONSHIP_TYPE.HYPERLINK, is_external=True
                    )
                    hyperlink = OxmlElement('w:hyperlink')
                    hyperlink.set(qn('r:id'), r_id)
                    new_run = OxmlElement('w:r')
                    rPr = OxmlElement('w:rPr')
                    rStyle = OxmlElement('w:rStyle')
                    rStyle.set(qn('w:val'), 'Hyperlink')
                    rPr.append(rStyle)
                    new_run.append(rPr)
                    t = OxmlElement('w:t')
                    t.text = weblink_value
                    new_run.append(t)
                    hyperlink.append(new_run)
                    para._p.append(hyperlink)
                else:
                    para.text = "N/A"
            if "{{sources}}" in para.text:
                # Build sources as hyperlinks if possible

                # Helper to add a hyperlink to a paragraph
                def add_hyperlink(paragraph, url):
                    # Create the w:hyperlink tag and add needed values
                    part = paragraph.part
                    r_id = part.relate_to(url, docx.opc.constants.RELATIONSHIP_TYPE.HYPERLINK, is_external=True)
                    hyperlink = OxmlElement('w:hyperlink')
                    hyperlink.set(qn('r:id'), r_id)
                    # Create a w:r element
                    new_run = OxmlElement('w:r')
                    # Create a w:rPr element
                    rPr = OxmlElement('w:rPr')
                    # Add color and underline for hyperlink style
                    rStyle = OxmlElement('w:rStyle')
                    rStyle.set(qn('w:val'), 'Hyperlink')
                    rPr.append(rStyle)
                    new_run.append(rPr)
                    # Create a w:t element and set the text
                    t = OxmlElement('w:t')
                    t.text = url
                    new_run.append(t)
                    hyperlink.append(new_run)
                    paragraph._p.append(hyperlink)

                # Remove the placeholder
                para.text = para.text.replace("{{sources}}", "")

                # Add each source as a hyperlink, stacked (one per line, no commas)
                sources = []
                for srcs in df['sources'].dropna().unique():
                    for src in [s.strip() for s in srcs.split(',') if s.strip()]:
                        sources.append(src)
                for i, src in enumerate(sources):
                    add_hyperlink(para, src)
                    if i < len(sources) - 1:
                        para.add_run().add_break()  # Add line break instead of comma

        # --- Save the document ---
        indicator_name = str(df['search_term'].iloc[0]) if 'search_term' in df.columns else 'Unnamed_Indicator'
        sanitized_name = indicator_name.replace(":", "_").replace("/", "_")

        current_date = datetime.now().strftime("%Y-%m-%d")
        folder_path = os.path.join(OUTPUT_DIR, current_date)
        os.makedirs(folder_path, exist_ok=True)

        output_path = os.path.join(folder_path, f"I&W_Report_{sanitized_name}.docx")
        doc.save(output_path)
        print(f"Saved report: {output_path}")

    except Exception as e:
        print(f"Error while generating report for {indicator_name}: {e}")

def main(vt_df, otx_df, recent_tags):
    # Normalize VT columns you rely on
    vt_df = vt_df.rename(columns={'ip': 'search_term', 'webLink': 'link'})

    # Normalize recent_tags to 'search_term' so we can join on one key
    if isinstance(recent_tags, pd.DataFrame) and not recent_tags.empty:
        recent_norm = recent_tags.rename(columns={'summary': 'search_term'}).copy()
        keep_cols = [c for c in ['search_term','observations','type','partners','observed_date','webLink'] if c in recent_norm.columns]
        recent_norm = recent_norm[keep_cols]
    else:
        recent_norm = pd.DataFrame(columns=['search_term'])

    # Combine VT and OTX
    combined_df = pd.merge(vt_df, otx_df, on='search_term', how='outer', suffixes=('_vt', '_otx'))

    # Consolidate sources
    sources_df = consolidate_sources(vt_df, otx_df)
    combined_df = pd.merge(combined_df, sources_df, on='search_term', how='left')

    # Merge recent tags (already on 'search_term')
    combined_df = pd.merge(combined_df, recent_norm, on='search_term', how='left')

    # Generate reports
    current_date = pd.Timestamp.now().strftime("%Y-%m-%d")
    for indicator in combined_df['search_term'].dropna().unique():
        indicator_df = combined_df[combined_df['search_term'] == indicator]
        sanitized = str(indicator).replace(":", "_").replace("/", "_")
        output_file = os.path.join(OUTPUT_DIR, f"I&W_Report_{sanitized}_{current_date}.docx")
        fill_word_template(TEMPLATE_PATH, output_file, indicator_df)

    
if __name__ == "__main__":
    main(vt_df, otx_df, recent_tags)



Saved report: C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports\2025-08-14\I&W_Report_103.124.94.57.docx
Saved report: C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports\2025-08-14\I&W_Report_114.98.227.36.docx
Saved report: C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports\2025-08-14\I&W_Report_121.152.230.177.docx
Saved report: C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports\2025-08-14\I&W_Report_125.88.174.211.docx
Saved report: C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports\2025-08-14\I&W_Report_159.203.36.189.docx
Saved report: C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports\2025-08-14\I&W_Report_166.186.196.135.docx
Saved report: C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports\2025-08-14\I&W_Report_190.14.253.66.docx
